# Read Data

In [0]:
spark.conf.set(
    "fs.azure.account.key.rstorage.dfs.core.windows.net", 
    "30j84K1fZv6zPumoR5DiXGQ+sNb8VLD0ikjb3W8mL8fONVfEMEo+iaEvpXTKytXTMkJ7onOq3225+AStZaqqUw=="
)

In [0]:
Sales_df = spark.read.format('parquet')\
    .option('inferSchema', True)\
        .load('abfss://bronze-adf@rgstorage.dfs.core.windows.net/')

In [0]:
Sales_df.display(10)

In [0]:
from pyspark.sql.functions import col, split, aggregate

In [0]:
Sales_df = Sales_df.withColumn('Model_Category', split(col('Model_ID'), '-')[0])

In [0]:
Sales_df.display()

In [0]:
from pyspark.sql.types import StringType

In [0]:
Sales_df.withColumn('Units_sold', col('Units_Sold').cast(StringType())).printSchema()

root
 |-- Branch_ID: string (nullable = true)
 |-- Dealer_ID: string (nullable = true)
 |-- Model_ID: string (nullable = true)
 |-- Revenue: long (nullable = true)
 |-- Units_sold: string (nullable = true)
 |-- Date_ID: string (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- BranchName: string (nullable = true)
 |-- DealerName: string (nullable = true)
 |-- Model_Category: string (nullable = true)



In [0]:
Sales_df = Sales_df.withColumn('Revenue_per_unit', col('Revenue')/col('Units_Sold'))
Sales_df.display()

In [0]:
from pyspark.sql import functions as F

# Transforming the raw data

In [0]:
Sales_agg = Sales_df.groupBy('Year', 'BranchName')\
    .agg(F.sum('Units_Sold').alias('Total_units'))\
        .sort('Year', 'Total_units', ascending=[1, 0])
Sales_agg.display()
                                                      

In [0]:
Sales_df.write.format('parquet')\
    .mode('overwrite')\
        .option('path', 'abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales')\
            .save()

# Querying the silver data

In [0]:
%sql
select * from parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales`